In [2]:
pip install langchain langchain_community langchain_chroma


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [4]:
import bs4
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

import getpass
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage


os.environ["GOOGLE_API_KEY"] = ''


llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")


from langchain_community.document_loaders import PyPDFLoader

file_path = r""
loader = PyPDFLoader(file_path)

docs = loader.load()

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)


vectorstore = Chroma(embedding_function=embeddings, persist_directory=None)
vectorstore.delete_collection()  
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory=None)


retriever = vectorstore.as_retriever(search_type="similarity",
    search_kwargs={"k": 100})

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (

    "Ти - студент, але є експертом з тієї теми, яку специфікована далі в контексті обернену в <> символи.",
    "Якщо користувач дає математичну задачу або завдання, ти повинен виокремити алгоритм з інформації наданої тобі.",
    "Алгоритм не повинен містити ніяких чисел, тільки процес. Перевір, чи містить твій алгоритм всі кроки так само, як в наданій тобі інформації.",
    "І після цього по цьому алгоритму ти виконуєш поставлену користувачем задачу, підставляючи числа.",
    "Рішення потрібно подавати у вигляді вирішеної задачі, де мінімально тексту, але по діях все розписано. Контекст:",
    "<{context}>"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

results = rag_chain.invoke({"input": "Зробити двi iтерацiї для знаходження найбiльшого кореня нелiнiйного рiвняння x3 + 4x − 6 = 0 методом дихотомiї. Записати умову припинення, ε = 0, 001."})

results

USER_AGENT environment variable not set, consider setting it to identify your requests.
c:\dev\smart-study-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.